# DTL: Thematic validation
This script is to conduct thematic validation of an LULC map that was generated on step 4 

In [ ]:
# !python -m pip install .. --quiet

import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

VERSION = 'v6'

provinces = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra_Provinces')
province_list = provinces.toList(provinces.size())
aoi = ee.Feature(province_list.get(0)).geometry() # aceh

# Load final LULC map from asset WITHOUT post-classification cleanup

final_map = ee.Image(f'projects/epistem2/assets/final_lulc_stack_Aceh_2020_{VERSION}')
validation_fc = ee.FeatureCollection(f'projects/epistem2/assets/Sumatra_Validation_Points').filterBounds(aoi)

In [ ]:
# Quick sanity check — fetches only the count, not all features
n_features = validation_fc.size().getInfo()
print(f"  → {n_features} validation features found")

first_feature   = validation_fc.first().getInfo()
prop_names      = list(first_feature["properties"].keys())
sample_props    = first_feature["properties"]
sample_geom     = first_feature["geometry"]["coordinates"]   # [lon, lat]

print("\n── Properties ───────────────────────────────")
print(f"  Names  : {prop_names}")
print(f"  Sample : {sample_props}")
print(f"  Coords : lon={sample_geom[0]:.4f}, lat={sample_geom[1]:.4f}")

print(final_map.projection().getInfo())
print(validation_fc.first().geometry().projection().getInfo())
print(validation_fc.size().getInfo())

In [ ]:
VALIDATION_CLASS_PROPERTY = 'IDe'   # e.g. "CLASS_ID"

raw_counts = (
    validation_fc
    .reduceColumns(
        reducer    = ee.Reducer.frequencyHistogram(),
        selectors  = [VALIDATION_CLASS_PROPERTY]
    )
    .get("histogram")
    .getInfo()
)

# Sort by class ID and print as a small table
print(f"\n── Count per class ({VALIDATION_CLASS_PROPERTY}) ────────────────")
print(f"  {'Class ID':<12}{'Count':>8}{'Share':>8}")
print(f"  {'─'*28}")
total = sum(raw_counts.values())
for cls_id, count in sorted(raw_counts.items(), key=lambda x: int(x[0])):
    share = count / total * 100
    print(f"  {cls_id:<12}{count:>8}{share:>7.1f}%")
print(f"  {'─'*28}")
print(f"  {'TOTAL':<12}{total:>8}")

# Run validation using luma_ge function

In [ ]:
from luma_ge.accuracy import (thematic_accuracy,sample_size_calculator,validation_error_flag,)

accuracy_assessor = thematic_accuracy()

success, results = accuracy_assessor.run_accuracy_assessment(
    lcmap           = final_map,
    validation_data = validation_fc,
    class_property  = VALIDATION_CLASS_PROPERTY,
    scale           = 100,
    confidence      = 0.95,
)
 
if not success:
    raise RuntimeError(f"Assessment failed: {results.get('error', 'unknown error')}")

In [ ]:
import numpy as np

summary = accuracy_assessor.format_accuracy_summary(results)
print("=== Thematic Accuracy Summary ===")
for name, value in summary.items():
    print(f"{name}: {value}")

print("\nProducer accuracy per class:", results['producer_accuracy'])
print("User accuracy per class:", results['user_accuracy'])
print("F1 scores per class:", results['f1_scores'])
print("Confusion matrix:\n", np.array(results['confusion_matrix']))